In [ ]:
# Imports
import cartopy
import cartopy.crs as ccrs
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import seaborn as sns
import yaml
from matplotlib import ticker
from matplotlib.lines import Line2D
from sklearn.metrics import mean_squared_error, r2_score
import sys
import yaml
from IPython.display import Markdown as md

import holoviews as hv
from holoviews import opts
import panel as pn
from bokeh.models import HoverTool #, Label
from bokeh.resources import INLINE
from bokeh.plotting import show
hv.notebook_extension('bokeh', 'matplotlib')
%matplotlib inline
data_show="both"

# variable info
variables = {}
variables['Latent_Heat'] = {'model': 'hfls', 'observations': 'LE_F_MDS_filtered', 'observations_lamda': lambda x: (x['LE_F_MDS_filtered']), "units": "W/m^2",
        "units_final": 'W/m^2', "input_attributes": ["hfls"], "model_lamda": lambda x: (x['hfls']), "show": data_show}

variables['Sensible_Heat'] = {'model': 'hfss', 'observations': 'H_F_MDS_filtered', 'observations_lamda': lambda x: (x['H_F_MDS_filtered']), "units": "W/m^2",
        "units_final": 'W/m^2', "input_attributes": ["hfss"], "model_lamda": lambda x: (x['hfss']), "show": data_show}

variables['Gross_Primary_Productivity'] = {'model': 'gpp', 'observations': 'GPP_DT_VUT_MEAN_filtered', 'observations_lamda': lambda x: (x['GPP_DT_VUT_MEAN_filtered'] * 1E3 * 60 * 60 * 24), 
        "units": "kg C/m^2/day", "units_final": 'gC/m^2/day', "input_attributes": ['gpp'], "model_lamda": lambda x: (x['gpp'] * 1E3 * 60 * 60 * 24), "show": data_show}

variables['Ecosystem_Respiration'] = {'model': 'reco', 'observations': 'RECO_DT_VUT_MEAN_filtered', 'observations_lamda': lambda x: (x['RECO_DT_VUT_MEAN_filtered'] * 1E3 * 60 * 60 * 24), 
        "units": "kg C/m^2/day", "units_final": 'gC/m^2/day', "input_attributes": ["ra", "rh"], "model_lamda": lambda x: ((x['ra'] + x['rh']) * 1E3 * 60 * 60 * 24), "show": data_show} # reco is 1*ra + 1*rh

variables['Net_Ecosystem_Productivity'] = {'model': 'nep', 'observations': 'NEE_VUT_MEAN_filtered', 'observations_lamda': lambda x: ((x['NEE_VUT_MEAN_filtered'] * -1) * 1E3 * 60 * 60 * 24),  
        "units": "kg C/m^2/day", "units_final": 'gC/m^2/day', "input_attributes": ["nep"], "model_lamda": lambda x: (x['nep'] * 1E3 * 60 * 60 * 24), "show": data_show} # "nee" needs to be inverted

In [ ]:
#this is the main function that accomplished plotting an example of it in use is below
def plot_site(site_name,\
data_path = '/space/hall5/sitestore/eccc/crd/ccrp/users/rsc001/sc_site_files/fluxnet_harvest_sites/site_runs/outputFiles/', \
obs_data_path = '/space/hall5/sitestore/eccc/crd/ccrp/users/rsc001/sc_site_files/fluxnet_harvest_sites/process_inputs/final/obs',\
yaml_data_path = '/home/rsc001/sc_file/git/developFLUXNETupdate/inputFiles/FLUXNETsitesCA',s_list=('both','both','both','both','both')\
,p_data=False,v_list=('Latent_Heat', 'Sensible_Heat', 'Gross_Primary_Productivity', 'Ecosystem_Respiration', 'Net_Ecosystem_Productivity')):

    
    model_data_path = data_path+site_name+"/netCDF"

    # Get info from site yamls
    def yaml_parse(file, att):
        try:
            with open(file, "r") as f:
                d = yaml.safe_load(f)
            if att == 'all':
                return d
            else:
                return(d[att])
        except:
            d = ""
            return d


    # Print all site info from yaml
    def yaml_print(yaml_file_path):
        for k,v in yaml_parse(yaml_file_path, "all").items():
            print(f'{k}: {v}')

    # Check if yaml exists and show full site name if available
    yaml_file_path = yaml_data_path + '/' + site_name + '/siteinfo.yaml'
    full_site_name = yaml_parse(yaml_file_path, "full_name")
    if full_site_name == '':
        print(f'The site info yaml file for {site_name} could not be parsed...')
        full_site_name = 'Site Info Unavailable'
        lat = None
        lon = None
    else:
        lat = yaml_parse(yaml_file_path, "lat")
        lon = yaml_parse(yaml_file_path, "lon")
    print("###"+site_name+" - "+full_site_name+"###")

    # Print other site info
    yaml_print(yaml_file_path)

    inner_index=0
    for var_to_plot in v_list:
        data_show=s_list[inner_index]

        # Read in the data
        full_site_name = 'Site Info Unavailable'
        if data_show != 'obs':
            model_data = None
            for v in variables[var_to_plot]['input_attributes']:
                try:
                    if full_site_name == 'Site Info Unavailable':
                        data = pd.read_csv(model_data_path + '/' + v + '_daily.csv', 
                                    nrows=1, usecols=['latitude', 'longitude'])
                        lat = list(data.latitude)[0]
                        lon = list(data.longitude)[0]
                    data = pd.read_csv(model_data_path + '/' + v + '_daily.csv', 
                                    usecols=["time", v], parse_dates=['time'])
                except:
                    data_show = 'obs'
                    print(f'The file {model_data_path}/{v}_daily.csv could not be read and parsed, replacing with empty dataset...')

                if model_data is None:
                    model_data = data
                else:
                    model_data = model_data.merge(data, how = 'left', left_on = 'time', right_on = 'time') 

        if data_show != 'model':
            try:
                obs_data_full = pd.read_csv(obs_data_path + '/' + site_name + '_obs.csv', usecols=['TIMESTAMP', 'DOY', variables[var_to_plot]["observations"]], 
                                                   parse_dates={'time':[0]}, skiprows=[1])
                obs_data_full[variables[var_to_plot]["observations"]] = obs_data_full[variables[var_to_plot]["observations"]].astype(float)

                if(var_to_plot == 'Gross_Primary_Productivity' or var_to_plot == 'Ecosystem_Respiration'):
                    obs_data_full[variables[var_to_plot]["observations"]].values[obs_data_full[variables[var_to_plot]["observations"]].values < 0] = np.nan
                if(p_data):
                    print(obs_data_full)

            except:
                if data_show == 'both':
                    data_show = 'model'
                else:
                    data_show = 'none'
                print(f'The file {obs_data_path}/{site_name}_obs.csv could not be read and parsed, replacing with empty dataset...')

            if (var_to_plot=='Latent_Heat'):    
                # Show site location
                if lat is not None and lon is not None:
                    plt.figure(figsize=(4*2, 4*3))
                    plot_crs = ccrs.Robinson(central_longitude=0)
                    ax = plt.axes(projection=plot_crs)
                    ax.set_global()
                    plt.scatter(x=lon, y=lat , s=200, color='deepskyblue', linewidth=1,  
                                marker='*',alpha=0.9,edgecolors='black', transform=ccrs.PlateCarree())
                    ax.coastlines('110m', linewidth=0.8, zorder=2,color='grey')
                    ax.set_extent([-180,180,-60,90],crs=ccrs.PlateCarree())
                    plt.title(f'{site_name} Location (lat: {lat}, lon: {lon})', size=16)
                else:
                    print(f'Location of {site_name} is unavailable...')

        # End notebook early if no data is found to plot
        class StopExecution(Exception):
            def _render_traceback_(self):
                pass

        if data_show == 'none':
            if full_site_name != 'Site Info Unavailable':
                yaml_print(yaml_file_path)
            raise StopExecution

        # Model Data Processing
        if data_show != 'obs':
            model_data = model_data.assign(temp_name = variables[var_to_plot]['model_lamda'])
            model_data.drop(variables[var_to_plot]['input_attributes'], axis = 1, inplace = True)
            model_data.rename(columns = {'temp_name':'Model'}, inplace = True)

            model_grouped = model_data.groupby(by=model_data['time'].dt.dayofyear).mean()
            # yrz = model_data['time'].iloc[-1].year - model_data['time'].iloc[0].year + 1
            model_grouped['DOY'] = model_grouped.index
            # model_grouped['Model Coverage'] = model_grouped.groupby(by=["DOY"]).sum()/yrz

        if data_show != 'model':
            # Observational Data Processing
            obs_data_full = obs_data_full.assign(temp_name = variables[var_to_plot]['observations_lamda'])
            obs_data_full.drop([variables[var_to_plot]['observations']], axis = 1, inplace = True)
            obs_data_full.rename(columns = {'temp_name':'Observation'}, inplace = True)

            obs_data = obs_data_full.drop(['DOY'], axis = 1)

            # Find out how many are not null.
            obs_grouped = obs_data_full
            obs_grouped['obsvar_NAN'] = obs_grouped['Observation'].notnull()
            # Group data first by day of year, keep the NaN values
            # NOTE: this needs Pandas to be >1.1 version!
            obs_grouped = obs_grouped.groupby(by=["DOY"], dropna=False).mean()
            # Find how many years we have
            yrz = obs_data_full['time'].iloc[-1].year - obs_data_full['time'].iloc[0].year + 1
            # Now figure out what percent of the total possible days actually have non-nan values
            obs_grouped['Observational Coverage'] = obs_grouped.groupby(by=["DOY"], dropna=False).sum()['obsvar_NAN']/yrz
            # Set the DOY as the index
            obs_grouped['DOY'] = obs_grouped.index

        if data_show == 'both':
            merged_data = obs_data.merge(model_data, how = 'left', left_on = 'time', right_on = 'time')
            merged_data.dropna(inplace = True)
            merged_data.drop('time', axis = 1, inplace = True)
            # print(merged_data)

        # Build Timeseries plot
        ### Still need to add the bounds if we want those
        timeseries = []
        if data_show != 'obs':
            timeseries.append(hv.Curve(model_data, label = 'Model'))
        if data_show != 'model':
            timeseries.append(hv.Curve(obs_data, label = 'Observation'))
        timeseries_plot_list = hv.Overlay(timeseries)
        title = site_name + ' - ' + var_to_plot + ' (' + variables[var_to_plot]['units_final'] + ') - Timeseries'
        timeseries_plot_list.opts(opts.Curve(height=400, width=800, line_width=1.50, line_alpha=0.6, tools=['hover'], 
                                             ylabel = variables[var_to_plot]['model'], title = title))
        show(hv.render(timeseries_plot_list))

        # Get stats for Crossplot
        def linear_regression(dataset):
            X = list(dataset.Observation)
            y = list(dataset.Model)

            # calculate parameteres, least_sqrs[0] is slope, least_sqrs[1] is y intercept
            least_sqrs = np.polyfit(X, y, 1)
            return least_sqrs[0], least_sqrs[1]

        # Build Crossplot
        if data_show == 'both':
            crossplot = [hv.Scatter(merged_data).opts(alpha = 0.6)]
            # Holoviews used np.polyfit() to calculate its Slope.from_scatter,
            # so the values calculated by this method as the same as those used to prodice the line in the plot
            lin_reg_slope, lin_reg_intercept = linear_regression(merged_data)
            r2 = r2_score(list(merged_data.Observation), list(merged_data.Model))
            rmse = mean_squared_error(list(merged_data.Observation), list(merged_data.Model), squared=False)
        #     print(list(merged_data.Observation))
        #     print(list(merged_data.Model))
            crossplot.append(hv.Slope.from_scatter(crossplot[0]))
            crossplot.append(hv.Slope(1, 0).opts(color='black', line_width=0.8))
            text = f'line eqn: y = {lin_reg_slope:.3f}x + {lin_reg_intercept:.3f} / R2 (coefficient of determination): {r2:.3f} / RMSE: {rmse:.3f}'
            crossplot.append(hv.Text(x = merged_data.Observation.max() * 0.5, y = merged_data.Model.max() * 1.05, 
                                     text = text).opts(color = 'steelblue', text_alpha = 0.8))
            crossplot_list = hv.Overlay(crossplot)
            title = site_name + ' - ' + var_to_plot + ' (' + variables[var_to_plot]['units_final'] + ') - Crossplot'
            crossplot_list.opts(height = 400, width = 800, tools = ['hover'], 
                                             title = title, xlim = (0, np.NaN),
                                             ylim = (0, np.NaN))

            crossplot_hv_object = hv.render(crossplot_list)
            show(crossplot_hv_object)
        # hv.help(hv.Text)

        # Build Seasonal Plot
        # Draw a scatter plot while assigning point colors and sizes to different
        # variables in the dataset
        f, ax = plt.subplots(figsize=(14, 6.5))
        sns.despine(f, left=True, bottom=True)

        if data_show != 'obs':
            sns.scatterplot(x="DOY", y='Model', #hue = 'Model Coverage', size = 'Model Coverage',
                            #palette="ch:r=-.2,d=.3",
                            sizes=10, linewidth=0,
                            data=model_grouped, ax=ax)

        if data_show != 'model':
            sns.scatterplot(x="DOY", y='Observation',
                            hue="Observational Coverage", size="Observational Coverage",
                            palette="ch:r=.2,d=.3", 
                            sizes=(1, 30), linewidth=0,
                            data=obs_grouped, ax=ax).set(ylabel = variables[var_to_plot]['model'], title = site_name + ' Seasonal Plot of Daily Coverage')

        # plt.legend(labels=['Observation', 'Model'])
        inner_index=inner_index+1

In [ ]:
#This is an example of the function used to plot a particular site. 
#It assumes the default arguments assigned in the function above.
plot_site(site_name="CA-Ca1",s_list=('model','model','both','both','both'))